# Build Semantic Search Over Your Browser Bookmarks — Colab / Kaggle / Binder companion

This notebook mirrors the local `uv` project from the course's
[Build Semantic Search Over Your Browser Bookmarks](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/bookmarks-semantic-search)
lesson, adapted to run in a hosted notebook with no local files: parsing a Netscape
bookmarks export, local embeddings with `sentence-transformers`, NumPy cosine-similarity
search, and a keyword-vs-semantic comparison.

See the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/bookmarks-semantic-search) for the full walkthrough and the
[local example project](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/bookmarks-semantic-search) for the real, file-based version of this same code — including how to run it against *your own* exported bookmarks.

## Step 0: Install dependencies

In [ ]:
!pip install sentence-transformers numpy

## Step 1: Sample bookmarks

A hosted notebook has no local `bookmarks.html` to read from, so a small sample export
matching the format Chrome/Firefox/Edge produce is embedded here directly as a string.
Folders are `<H3>` tags, bookmarks are `<A>` tags with the URL in `HREF`.

In [ ]:
BOOKMARKS_HTML = """<!DOCTYPE NETSCAPE-Bookmark-file-1>
<DL><p>
    <DT><H3>Bookmarks bar</H3>
    <DL><p>
        <DT><A HREF="https://github.com/">GitHub</A>
        <DT><A HREF="https://docs.python.org/3/">Python documentation</A>
        <DT><A HREF="https://abderrahim-lectures.github.io/python-data-analysis-course/">PYDA course website</A>
    </DL><p>
    <DT><H3>Machine Learning</H3>
    <DL><p>
        <DT><A HREF="https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html">scikit-learn: train_test_split docs</A>
        <DT><A HREF="https://scikit-learn.org/stable/modules/ensemble.html">Ensemble methods — Random forests</A>
        <DT><A HREF="https://pytorch.org/tutorials/">PyTorch tutorials</A>
        <DT><A HREF="https://huggingface.co/models">Hugging Face model hub</A>
        <DT><A HREF="https://colab.research.google.com/">Google Colab — free notebooks</A>
    </DL><p>
    <DT><H3>Web Development</H3>
    <DL><p>
        <DT><A HREF="https://developer.mozilla.org/en-US/docs/Web">MDN Web Docs</A>
        <DT><A HREF="https://react.dev/">React — UI library docs</A>
        <DT><A HREF="https://fastapi.tiangolo.com/">FastAPI — Python web framework</A>
    </DL><p>
    <DT><H3>Databases</H3>
    <DL><p>
        <DT><A HREF="https://www.postgresql.org/docs/">PostgreSQL docs</A>
        <DT><A HREF="https://www.sqlite.org/docs.html">SQLite docs</A>
        <DT><A HREF="https://pandas.pydata.org/docs/">pandas documentation</A>
    </DL><p>
</DL><p>
"""


## Step 2: Parse the export into records

Same parser as [`parse_bookmarks.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/bookmarks-semantic-search/parse_bookmarks.py):
`html.parser` walks the nested `<DL>/<H3>/<A>` structure, pushing folder names onto a stack
so every bookmark gets a `folder` path like `Machine Learning/Models`.

In [ ]:
from html.parser import HTMLParser


class BookmarkParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.folder_stack: list[str] = []
        self.pending_folder: str | None = None
        self.records: list[dict] = []

    def handle_starttag(self, tag: str, attrs) -> None:
        attrs = dict(attrs)
        if tag == "h3":
            self.pending_folder = ""
        elif tag == "dl":
            if self.pending_folder is not None:
                name = " ".join(self.pending_folder.split())
                self.folder_stack.append(name)
                self.pending_folder = None
            else:
                self.folder_stack.append("")
        elif tag == "a":
            self._pending = {"href": attrs.get("href", ""), "text": ""}

    def handle_endtag(self, tag: str) -> None:
        if tag == "dl" and self.folder_stack:
            self.folder_stack.pop()
        elif tag == "a" and hasattr(self, "_pending"):
            title = " ".join(self._pending["text"].split()).strip()
            record = {
                "title": title or self._pending["href"],
                "url": self._pending["href"],
                "folder": "/".join(name for name in self.folder_stack if name),
            }
            if record["url"]:
                self.records.append(record)
            del self._pending

    def handle_data(self, data: str) -> None:
        if self.pending_folder is not None:
            self.pending_folder += data
        elif hasattr(self, "_pending"):
            self._pending["text"] += data


parser = BookmarkParser()
parser.feed(BOOKMARKS_HTML)
records = parser.records
print(f"Parsed {len(records)} bookmarks")
for r in records[:4]:
    print(f"  [{r['folder']}] {r['title']} -> {r['url']}")

## Step 3: Embed every bookmark locally

Same model as [`build_index.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/bookmarks-semantic-search/build_index.py):
`all-MiniLM-L6-v2`, run entirely on CPU, no API key needed. Unlike the notes/RAG projects
there is no chunking — each bookmark (title) is already the atomic record.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Embedding {len(records)} bookmarks with {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
texts = [r["title"] for r in records]
embeddings = model.encode(texts, normalize_embeddings=True)

print(f"Embedded {embeddings.shape[0]} bookmarks ({embeddings.shape[1]}-dim)")

## Step 4: Search by meaning

Same cosine-similarity search as [`search.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/bookmarks-semantic-search/search.py) —
normalized embeddings make cosine similarity a plain dot product. Notice the query shares
almost no words with the titles it surfaces.

In [ ]:
def search(query: str, top_k: int = 5) -> list[dict]:
    query_vector = model.encode([query], normalize_embeddings=True)[0]
    similarities = embeddings @ query_vector
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {**records[i], "score": float(similarities[i])}
        for i in top_indices
    ]


for r in search("how do I split my data into train and test sets"):
    print(f"{r['score']:.3f}  [{r['folder']}] {r['title']}")
    print(f"      {r['url']}")

## Step 5: Compare against keyword search

The same query through a naive keyword ranker (counts query words that appear in the
title) vs. the embedding search — the concrete demonstration of where each wins.

In [ ]:
def keyword_search(query: str, top_k: int = 5) -> list[dict]:
    words = [w.lower() for w in query.split() if len(w) > 2]
    scored = []
    for record in records:
        title_lower = record["title"].lower()
        hits = sum(1 for w in words if w in title_lower)
        if hits:
            scored.append({**record, "score": hits})
    scored.sort(key=lambda r: r["score"], reverse=True)
    return scored[:top_k]


query = "scikit learn train test split"
print("Keyword search:")
for r in keyword_search(query):
    print(f"  {r['score']} hit(s)  [{r['folder']}] {r['title']}")

print("\nSemantic search:")
for r in search(query):
    print(f"  {r['score']:.3f}  [{r['folder']}] {r['title']}")